# Phase 0: Reproduce GCR Baseline

**Goal:** Reproduce GCR's Hits@1 >= 91% on WebQSP.

**Requires:** A100 GPU (40GB VRAM) for the 8B LLM at full precision.
With 4-bit quantization a T4/V100 may work.

**Pipeline:**
1. Step 1: `predict_paths_and_answers.py` — constrained decoding with KG-Trie
2. Step 2: `predict_final_answer.py` — answer extraction (optional)

## 1. Colab / Local Environment Setup

In [2]:
import sys, os, json, warnings, gc, subprocess
import numpy as np
import torch

IN_COLAB = 'google.colab' in sys.modules
print(f"Python: {sys.version}")
print(f"Running in Colab: {IN_COLAB}")

# Check GPU
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 20:
        print("WARNING: <20GB VRAM. Use 4-bit quantization or reduce num_beams.")
else:
    print("WARNING: No GPU detected. This pipeline requires a GPU for the 8B model.")

if IN_COLAB:
    if not os.path.exists('dca-trie'):
        !git clone https://github.com/adjanour/dca-trie.git
        %cd dca-trie
        !git submodule update --init
    else:
        %cd dca-trie
    !pip install -q -e .
else:
    print("Running locally. Ensure poetry env is active.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Running in Colab: True
GPU: NVIDIA A100-SXM4-40GB  VRAM: 42.4 GB
Cloning into 'dca-trie'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 136 (delta 44), reused 127 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.88 MiB | 50.63 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/dca-trie
Submodule 'vendor/gcr' (https://github.com/RManLuo/graph-constrained-reasoning.git) registered for path 'vendor/gcr'
Cloning into '/content/dca-trie/vendor/gcr'...
Submodule path 'vendor/gcr': checked out '9518e8e69197d5e577a6e414060fb898771617b9'
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69

## 2. Set HuggingFace Token

The GCR model is gated.
Token: https://huggingface.co/settings/tokens
Access: https://huggingface.co/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

In [5]:
from huggingface_hub import login
from google.colab import userdata


HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF_TOKEN configured.")
else:
    raise ValueError("HF_TOKEN required. Set it above or in .env")

HF_TOKEN configured.


## 3. Step 1: Run GCR Path Prediction

Constrained decoding with KG-Trie on 100 questions.
Takes ~45 min on A100.

**Flags you may want to tune:**
- `--n 100`: number of questions (use 10 for a quick smoke test first)
- `--model_name gcr-Llama-2-7b-chat-hf`: GCR's original smaller model
- `--attn_implementation flash_attention_2`: requires flash-attn installed
- `--load_in_4bit`: use 4-bit quantization (needs less VRAM)

In [ ]:
%%time
!python gcr/workflow/predict_paths_and_answers.py \
    --model_name rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
    --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
    --data_path rmanluo \
    --d RoG-webqsp \
    --split test[:100] \
    --predict_path results/GenPaths \
    --n 1 \
    --max_new_tokens 128 \
    --k 10 \
    --generation_mode beam \
    --index_path_length 2 \
    --dtype bf16 \
    --attn_implementation sdpa

print("Step 1 complete.")

2026-05-13 03:02:00.568627: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 03:02:02.032059: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Save results to:  results/GenPaths/RoG-webqsp/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct/test[:100]/zero-shot-beam-k10-index_len2
Prepare pipline for inference...
Loading checkpoint shards: 100% 4/4 [00:56<00:00, 14.16s/it]
generation_config.json: 100% 184/184 [00:00<00:00, 1.05MB/s]
  0% 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/co

## 4. Check Results

In [ ]:
import glob
import os
import pandas as pd

# The base directory where results are stored
base_output_dir = "/content/dca-trie/results/GenPaths/RoG-webqsp/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct/"

# The problematic directory name with special characters
split_name = "test[:100]"
# Use glob.escape() to properly escape the special characters for glob.glob
escaped_split_name = glob.escape(split_name)

# Construct the full glob pattern using the escaped name
glob_pattern = os.path.join(base_output_dir, escaped_split_name, "zero-shot-beam-k10-index_len2", "predictions.jsonl")

# Now use glob.glob with the correctly escaped pattern
pred_files = glob.glob(glob_pattern)
print(f"Found {len(pred_files)} prediction files")

if pred_files:
    df = pd.read_json(pred_files[0], lines=True)
    print(f"\nPredictions: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nSample:")
    print(df.iloc[0].to_dict() if len(df) > 0 else "empty")

Found 1 prediction files

Predictions: 100
Columns: ['id', 'question', 'prediction', 'ground_truth', 'ground_truth_paths', 'input']

Sample:
{'id': 'WebQTest-0', 'question': 'what does jamaican people speak', 'prediction': ['# Reasoning Path:\nJamaica -> location.country.languages_spoken -> Jamaican Creole English Language\n# Answer:\nJamaican Creole English Language', '# Reasoning Path:\nJamaica -> location.country.languages_spoken -> Jamaican English\n# Answer:\nJamaican English', '# Reasoning Path:\nJamaica -> location.country.form_of_government -> Parliamentary system -> government.form_of_government.countries -> Belize\n# Answer:\nBelize', '# Reasoning Path:\nJamaica -> location.country.form_of_government -> Parliamentary system -> government.form_of_government.countries -> Bahamas\n# Answer:\nBahamas', '# Reasoning Path:\nJamaica -> location.country.languages_spoken -> Jamaican Creole English Language\n# Answer:\nJamaian Creole English Language', '# Reasoning Path:\nJamaica -> lo

## 5. Evaluate Hits@1 and F1

In [ ]:
if pred_files:
    reasoning_path = pred_files[0]
    print(f"Reasoning paths: {reasoning_path}")
    !python dca-trie/gcr/workflow/predict_final_answer.py \
        --data_path rmanluo \
        --d RoG-webqsp \
        --split test[:100] \
        --predict_path results/KGQA \
        --add_path true \
        --reasoning_path {reasoning_path} \
        --model_name rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
        --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
        --attn_implementation sdpa \
        --k 10 \
        --generation_mode beam \
        --dtype bf16
else:
    print("No prediction files found. Run Step 1 first.")

Reasoning paths: /content/dca-trie/results/GenPaths/RoG-webqsp/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct/test[:100]/zero-shot-beam-k10-index_len2/predictions.jsonl
2026-05-13 03:44:53.798921: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 03:44:53.901178: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Map: 100% 100/100 [00:00<00:00, 1122.49 examples/s]
Save results to:  results/KGQA/RoG-webqsp/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct/test[:100]/add_path__content_dca-trie_results_GenPaths_RoG-webqsp_rmanluo_GCR-Me

## Phase 0 Exit Criteria
- [ ] Step 1 runs without errors
- [ ] Hits@1 >= 91% on 100 questions
- [ ] Predictions saved to `results/GenPaths/.../predictions.jsonl`

In [7]:
!python experiments/threshold_sweep_v1.py --num 100 --dataset webqsp

Loading SemanticScorer (MiniLM)...
Loading weights: 100% 103/103 [00:00<00:00, 972.96it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading webqsp from HuggingFace...
  Loaded 100 questions
  MID coverage: 37022/37070 (99.9%)

Sweeping tau from 0.1 to 0.6...
tau    FNR      Reduction    Filtered   Original   SIR     
------------------------------------------------------------
Measuring baseline SIR on unfiltered tries...
object address  : 0x79bc6f9e7e20
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


In [ ]:
!python experiments/threshold_sweep_v1.py --num 100 --dataset webqsp --tau_min 0.55

Loading SemanticScorer (MiniLM)...
Loading weights: 100% 103/103 [00:00<00:00, 1029.96it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading webqsp from HuggingFace...
  Loaded 100 questions
  MID coverage: 37022/37070 (99.9%)

Sweeping tau from 0.4 to 0.6...
tau    FNR      Reduction    Filtered   Original   SIR     
------------------------------------------------------------
Measuring baseline SIR on unfiltered tries...
  tau=0.40  FNR=0.010  reduction=33.6%  filtered=1562  original=2553  SIR=0.2887  baseline SIR=0.2887
  tau=0.45  FNR=0.010  reduction=50.1%  filtered=1102  original=2553  SIR=0.2887  baseline SIR=0.2887
  tau=0.50  FNR=0.040  reduction=66.1%  filtered=692